# End-to-end condition-to-similarity demo

Enter a retrieval condition and choose the table containing the entities you want to compare. The notebook ranks only signals rooted at that table, lets you inspect and weight compatible signals, then computes a similarity ranking. Semantic text vectors are reused from `/storage/lcostes/simbench-cache/text-embeddings.sqlite3`; precompute once with `python scripts/precompute_text_embeddings.py`.


In [1]:
from __future__ import annotations

import json
import importlib
import os
import subprocess
import sys
import tempfile
import traceback
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import Markdown, clear_output, display

REPO = Path.cwd().resolve()
if not (REPO / "src").is_dir():
    REPO = Path("/storage/lcostes/Similarity_benchmark")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

SIMBENCH_ENV = Path("/storage/lcostes/ray-share/venvs/simbench")
DATASET_ROOT = REPO / "data/dev/rel-amazon"
SIMBENCH_TEMP_ROOT = Path("/storage/lcostes/tmp/simbench-e2e")
SIMBENCH_TEMP_ROOT.mkdir(parents=True, exist_ok=True)
OCI_COMPARTMENT = os.environ.get("OCI_GENAI_COMPARTMENT", "").strip()
MODEL_CACHE_ROOT = Path(os.environ.get("SIMBENCH_MODEL_CACHE_ROOT", "/storage/lcostes/model-cache/huggingface/hub"))

from src.datasets.local_relbench_dataset import LocalRelBenchDataset
from src.schema.schema_graph import SchemaGraph
from src.signals.candidates import candidate_from_dict
import src.similarity.signal_measures as signal_measures
signal_measures = importlib.reload(signal_measures)
DEFAULT_TEXT_EMBEDDING_CACHE = signal_measures.DEFAULT_TEXT_EMBEDDING_CACHE
SignalMeasureExecutor = signal_measures.SignalMeasureExecutor
TEXT_EMBEDDING_CACHE = Path(os.environ.get("SIMBENCH_TEXT_EMBEDDING_CACHE", str(DEFAULT_TEXT_EMBEDDING_CACHE)))

dataset = LocalRelBenchDataset(DATASET_ROOT)
graph = SchemaGraph.from_manifest(dataset.schema)
executor = SignalMeasureExecutor(dataset, graph, model_cache_root=MODEL_CACHE_ROOT, text_embedding_cache=TEXT_EMBEDDING_CACHE)

DEFAULT_MEASURE_VARIANT = "auto"  # Change this to set the initial measure for every selected signal.
MEASURE_LABELS = {
    "auto": "",
    "jensen-shannon": "Jensen–Shannon",
    "hellinger": "Hellinger",
    "wl-subtree-kernel": "WL subtree kernel",
    "diffusion-distance": "One-step diffusion distance",
}
_MEASURE_OPTIONS_CACHE = {}
_AUTO_MEASURE_LABEL_CACHE = {}
def measure_options_for(candidate):
    if candidate.id not in _MEASURE_OPTIONS_CACHE:
        _MEASURE_OPTIONS_CACHE[candidate.id] = executor.available_measure_variants(candidate)
    if candidate.id not in _AUTO_MEASURE_LABEL_CACHE:
        _AUTO_MEASURE_LABEL_CACHE[candidate.id] = executor.automatic_measure_name(candidate)
    return [
        (f"{_AUTO_MEASURE_LABEL_CACHE[candidate.id]} (auto)" if key == "auto" else MEASURE_LABELS[key], key)
        for key in _MEASURE_OPTIONS_CACHE[candidate.id]
    ]

MODEL_EFFORTS = {
    "gpt-6-luna": ("none", "low", "medium", "high", "xhigh"),
    "gpt-5.6-terra": ("none", "low", "medium", "high"),
    "gpt-6-sol": ("none", "low", "medium", "high"),
    "gpt-6-astra": ("low", "medium", "high"),
}

def entity_ids(table):
    primary_key = dataset.get_primary_key(table)
    if primary_key is None:
        return tuple(range(dataset.get_table(table).num_rows)), "row position"
    return tuple(dataset.get_table(table)[primary_key].to_pylist()), "primary key"

def entity_row(table, entity_id):
    primary_key = dataset.get_primary_key(table)
    if primary_key is None:
        return dataset.get_table(table).slice(int(entity_id), 1).to_pylist()[0]
    values = dataset.get_table(table)[primary_key].to_pylist()
    resolved = next(value for value in values if str(value) == str(entity_id))
    return dataset.get_entity(table, resolved).to_pylist()[0]

def discovery_score_value(stored):
    value = stored.get("utility_score", stored.get("score"))
    return float(value) if isinstance(value, (int, float)) and not isinstance(value, bool) else None

def discovery_score_label(stored, score_type):
    if "utility_score" in stored:
        scale = "10" if score_type.endswith("_0_10") else "100" if score_type.endswith("_0_100") else "?"
        return f"discovery score {stored['utility_score']}/{scale}"
    if "score" in stored:
        return f"model score {float(stored['score']):.3g}"
    return "score unavailable"


## 1. Choose the condition and entity table, then discover signals

Discovery is restricted to signals rooted at the selected entity table. Changing the table after discovery clears the old results; run discovery again for the new table.


In [2]:
entity_picker = widgets.Dropdown(options=dataset.get_tables(), value="product", description="Compare entities from:", style={"description_width": "initial"})
condition_box = widgets.Textarea(
    value="Find products that serve similar purposes, even if their descriptions use different terminology.",
    description="Condition:",
    layout=widgets.Layout(width="95%", height="90px"),
)
model_picker = widgets.Dropdown(options=tuple(MODEL_EFFORTS), value="gpt-6-luna", description="Discovery model:", style={"description_width": "initial"})
effort_picker = widgets.Dropdown(options=MODEL_EFFORTS[model_picker.value], value="none", description="Reasoning effort:", style={"description_width": "initial"})
max_tokens_picker = widgets.IntText(value=2048, description="Max output tokens:", style={"description_width": "initial"})
discover_button = widgets.Button(description="Discover signals", button_style="primary", icon="search")
discovery_status = widgets.HTML()
discovery_output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px"))

CURRENT_DISCOVERY = None
CURRENT_WORKDIR = None

def refresh_efforts(*_):
    efforts = MODEL_EFFORTS[model_picker.value]
    effort_picker.options = efforts
    if effort_picker.value not in efforts:
        effort_picker.value = efforts[0]

model_picker.observe(refresh_efforts, names="value")

discovery_controls = widgets.VBox([
    entity_picker,
    condition_box,
    widgets.HBox([model_picker, effort_picker, max_tokens_picker]),
    discover_button,
    discovery_status,
    discovery_output,
])
display(discovery_controls)


## 2. Choose entities, then select signals, measures, and weights

Choose the table containing the entities you want to compare and retrieve. Each selected signal gets its own measure dropdown, containing only compatible choices. The automatic option names the function it will use for that signal (for example, `Percentile Wasserstein similarity (auto)`). Change `DEFAULT_MEASURE_VARIANT` near the top of the setup cell to set the initial choice for every signal. Numeric and temporal distribution variants appear only for multi-value signals; graph variants are available only for relationship signals. Diffusion may return missing scores when there are no shared neighbors.


In [3]:
query_picker = widgets.Dropdown(description="Query entity ID:", layout=widgets.Layout(width="60%"))
query_preview = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px"))
signal_picker = widgets.SelectMultiple(description="Discovered signals:", rows=10, layout=widgets.Layout(width="95%"))
measure_controls = widgets.VBox()
measure_pickers = {}
weight_mode = widgets.ToggleButtons(
    options=[("Equal weights (recommended)", "equal"), ("Relative discovery scores", "discovery"), ("Custom sliders", "custom")],
    value="equal", description="Weight preset:", style={"description_width": "initial"},
)
weight_help = widgets.HTML()
weight_controls = widgets.VBox()
score_button = widgets.Button(description="Find nearest entities", button_style="success", icon="play", disabled=True)
score_status = widgets.HTML()
score_output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px"))
weight_sliders = {}
_updating_weights = False

def show_query_entity(*_):
    with query_preview:
        clear_output(wait=True)
        if query_picker.value in (None, ""):
            display(Markdown("Select an entity table and query ID to preview it."))
            return
        row = entity_row(entity_picker.value, query_picker.value)
        display(Markdown(f"### Current query entity: `{entity_picker.value}` / `{query_picker.value}`"))
        display(pd.DataFrame([{"attribute": key, "value": value} for key, value in row.items()]))

def refresh_query_ids(*_):
    values, id_kind = entity_ids(entity_picker.value)
    query_picker.options = [str(value) for value in values]
    query_picker.value = str(values[0]) if values else None
    score_status.value = f"<b>{len(values):,}</b> {entity_picker.value} entities are available; query IDs use the table's <b>{id_kind}</b>."
    show_query_entity()

def selected_records():
    if CURRENT_DISCOVERY is None:
        return {}, ""
    score_type = str(CURRENT_DISCOVERY.get("score_type", ""))
    records = {
        item["id"]: item for item in CURRENT_DISCOVERY.get("signals", [])
        if isinstance(item, dict) and item.get("source_table") == entity_picker.value
    }
    return records, score_type

def update_weight_help():
    if weight_mode.value == "equal":
        weight_help.value = "All selected signals have weight 1."
    elif weight_mode.value == "discovery":
        weight_help.value = "Each weight is relative to the highest selected discovery score, from 1 to 10."
    else:
        weight_help.value = "Set each selected signal from 0 to 10. A weight of 0 removes its contribution."

def apply_weight_mode(*_):
    global _updating_weights
    _updating_weights = True
    try:
        records, _ = selected_records()
        if weight_mode.value == "equal":
            values = {signal_id: 1 for signal_id in weight_sliders}
        elif weight_mode.value == "discovery":
            scores = {signal_id: discovery_score_value(records.get(signal_id, {})) for signal_id in weight_sliders}
            maximum = max((value for value in scores.values() if value is not None and value > 0), default=0.0)
            values = {signal_id: max(1, min(10, round(10 * value / maximum))) if value is not None and maximum else 1 for signal_id, value in scores.items()}
        else:
            values = {}
        for signal_id, value in values.items():
            weight_sliders[signal_id].value = value
        update_weight_help()
    finally:
        _updating_weights = False

def slider_changed(_):
    if not _updating_weights and weight_mode.value != "custom":
        weight_mode.value = "custom"

def rebuild_weight_controls(*_):
    global weight_sliders, measure_pickers
    old_values = {signal_id: slider.value for signal_id, slider in weight_sliders.items()}
    old_measures = {signal_id: picker.value for signal_id, picker in measure_pickers.items()}
    records, score_type = selected_records()
    weight_sliders = {}
    measure_pickers = {}
    controls = []
    measure_rows = []
    for signal_id in signal_picker.value:
        stored = records.get(signal_id)
        if stored is None:
            continue
        candidate = candidate_from_dict(stored)
        choices = measure_options_for(candidate)
        allowed_measures = [value for _, value in choices]
        selected_measure = old_measures.get(signal_id, DEFAULT_MEASURE_VARIANT)
        if selected_measure not in allowed_measures:
            selected_measure = "auto"
        measure_picker = widgets.Dropdown(
            options=choices, value=selected_measure, description="Measure:",
            style={"description_width": "initial"}, layout=widgets.Layout(width="95%"),
        )
        measure_pickers[signal_id] = measure_picker
        measure_rows.append(widgets.VBox([
            widgets.HTML(f"<b>#{stored.get('rank', '?')}</b> · <code>{signal_id}</code>"), measure_picker,
        ]))
        slider = widgets.IntSlider(value=old_values.get(signal_id, 1), min=0, max=10, step=1, description="weight:", layout=widgets.Layout(width="55%"), style={"description_width": "initial"})
        slider.observe(slider_changed, names="value")
        weight_sliders[signal_id] = slider
        controls.append(widgets.VBox([
            widgets.HTML(f"<b>#{stored.get('rank', '?')}</b> · {discovery_score_label(stored, score_type)}<br><code>{signal_id}</code>"),
            slider,
        ]))
    measure_controls.children = tuple(measure_rows) or (widgets.HTML("Run discovery, then select signals to choose their measures."),)
    weight_controls.children = tuple(controls) or (widgets.HTML("Run discovery, then select one or more compatible signals."),)
    apply_weight_mode()

def populate_discovered_signals():
    records, score_type = selected_records()
    sorted_records = sorted(records.values(), key=lambda row: row.get("rank", 10**9))
    signal_picker.options = [
        (f"#{stored.get('rank', '?')} | {discovery_score_label(stored, score_type)} | {stored['id']}", stored["id"])
        for stored in sorted_records
    ]
    signal_picker.value = tuple(stored["id"] for stored in sorted_records[:3])
    rebuild_weight_controls()

def refresh_entity_selection(*_):
    global CURRENT_DISCOVERY, CURRENT_WORKDIR
    refresh_query_ids()
    if CURRENT_DISCOVERY is not None:
        CURRENT_DISCOVERY = None
        CURRENT_WORKDIR = None
        signal_picker.options = []
        signal_picker.value = ()
        rebuild_weight_controls()
        score_button.disabled = True
        with score_output:
            clear_output(wait=True)
        score_status.value = ""
        discovery_status.value = "Entity table changed. Run discovery again to rank signals for this table."

def current_weights():
    return {signal_id: slider.value for signal_id, slider in weight_sliders.items()}

entity_picker.observe(refresh_entity_selection, names="value")
query_picker.observe(show_query_entity, names="value")
signal_picker.observe(rebuild_weight_controls, names="value")
weight_mode.observe(apply_weight_mode, names="value")
refresh_query_ids()

signal_controls = widgets.VBox([
    entity_picker,
    query_picker,
    query_preview,
    signal_picker,
    measure_controls,
    weight_mode,
    weight_help,
    weight_controls,
    score_button,
    score_status,
    score_output,
])
display(signal_controls)


In [4]:
def run_discovery(_button):
    global CURRENT_DISCOVERY, CURRENT_WORKDIR
    condition = condition_box.value.strip()
    if not condition:
        discovery_status.value = "<span style='color:#b91c1c'>Enter a condition first.</span>"
        return
    if max_tokens_picker.value <= 0:
        discovery_status.value = "<span style='color:#b91c1c'>Max output tokens must be positive.</span>"
        return
    if not OCI_COMPARTMENT:
        discovery_status.value = "<span style='color:#b91c1c'>Set OCI_GENAI_COMPARTMENT in the notebook kernel environment first.</span>"
        return
    discover_button.disabled = True
    discovery_status.value = f"Ranking signals rooted at {entity_picker.value}. This can take a little while for the selected OCI model…"
    with discovery_output:
        clear_output(wait=True)
        try:
            workdir = Path(tempfile.mkdtemp(prefix="run-", dir=SIMBENCH_TEMP_ROOT))
            conditions_path = workdir / "condition.jsonl"
            output_path = workdir / "discovery.jsonl"
            conditions_path.write_text(json.dumps({"id": "custom-condition", "condition": condition}) + "\n")
            command = [
                "conda", "run", "--no-capture-output", "-p", str(SIMBENCH_ENV), "python",
                "scripts/discover_signals.py",
                "--conditions", str(conditions_path),
                "--dataset-root", str(DATASET_ROOT),
                "--method", "catalog-rank",
                "--model", model_picker.value,
                "--reasoning-effort", effort_picker.value,
                "--utility-scale", "0-10",
                "--max-tokens", str(max_tokens_picker.value),
                "--hop-limit", "2",
                "--catalog-view", "path-aware",
                "--source-table", entity_picker.value,
                "--output", str(output_path),
            ]
            run_env = os.environ.copy()
            run_env["OCI_GENAI_COMPARTMENT"] = OCI_COMPARTMENT
            display(Markdown(f"Running **catalog-rank / {model_picker.value}** on path-aware signals rooted at **{entity_picker.value}**…"))
            completed = subprocess.run(command, cwd=REPO, env=run_env, text=True, capture_output=True)
            if completed.stdout.strip():
                print(completed.stdout)
            if completed.returncode != 0:
                if completed.stderr.strip():
                    print(completed.stderr)
                raise RuntimeError(f"Signal discovery exited with code {completed.returncode}.")
            CURRENT_DISCOVERY = json.loads(output_path.read_text().splitlines()[0])
            CURRENT_WORKDIR = workdir
            populate_discovered_signals()
            discovery_status.value = f"Discovery complete: {len(CURRENT_DISCOVERY['signals'])} signals ranked for {entity_picker.value}. Select signals below."
            score_button.disabled = not bool(signal_picker.value)
        except Exception as error:
            traceback.print_exc()
            discovery_status.value = f"<span style='color:#b91c1c'><b>Discovery error:</b> {error}</span>"
        finally:
            discover_button.disabled = False

def score_nearest(_button):
    if CURRENT_DISCOVERY is None:
        score_status.value = "<span style='color:#b91c1c'>Run signal discovery first.</span>"
        return
    score_button.disabled = True
    score_status.value = "Materializing selected paths and scoring candidate entities…"
    with score_output:
        clear_output(wait=True)
        try:
            records, _ = selected_records()
            selected_ids = list(signal_picker.value)
            if not selected_ids:
                raise ValueError("Select at least one discovered signal.")
            candidates = [candidate_from_dict(records[signal_id]) for signal_id in selected_ids]
            materialized = [executor.materialize(candidate, measure_variant=measure_pickers[candidate.id].value) for candidate in candidates]
            ranked = executor.score_query(query_picker.value, materialized, weights=current_weights())
            query_id = ranked[0].query_entity_id if ranked else query_picker.value
            display(Markdown(f"## Nearest {entity_picker.value} entities for `{query_id}`"))
            evidence = pd.DataFrame([{
                "rank": records[signal.candidate.id].get("rank"),
                "signal": signal.candidate.id,
                "meaning": signal.candidate.explain_for_human(graph),
                "measure": signal.measure,
                "weight": current_weights().get(signal.candidate.id, 1),
            } for signal in materialized]).sort_values("rank")
            display(evidence)
            rows, component_rows = [], []
            for rank, pair in enumerate(ranked[:20], start=1):
                available = [component for component in pair.components if component.get("score") is not None]
                rows.append({
                    "rank": rank,
                    "candidate entity": pair.candidate_entity_id,
                    "combined similarity": pair.similarity_score,
                    "available signals": f"{len(available)}/{len(pair.components)}",
                })
                component_rows.extend({
                    "rank": rank,
                    "candidate entity": pair.candidate_entity_id,
                    "signal": component["signal_id"],
                    "component similarity": component["score"],
                    "evidence confidence": component.get("confidence", 1.0),
                    "effective weight": component.get("effective_weight", component.get("weight", 1.0)),
                } for component in available)
            display(pd.DataFrame(rows).round(3))
            display(Markdown("### Available component scores"))
            display(pd.DataFrame(component_rows).round(3) if component_rows else pd.DataFrame({"note": ["No selected signal had values for the returned entities."]}))
            display(Markdown("### Nearest entity records"))
            display(pd.DataFrame([
                {"candidate entity": pair.candidate_entity_id, "similarity": pair.similarity_score, **{key: str(value)[:240] for key, value in entity_row(entity_picker.value, pair.candidate_entity_id).items()}}
                for pair in ranked[:20]
            ]).round(3))
            score_status.value = f"Scored {len(ranked):,} candidates with {len(materialized)} selected signal(s)."
        except Exception as error:
            traceback.print_exc()
            score_status.value = f"<span style='color:#b91c1c'><b>Similarity error:</b> {error}</span>"
            display(Markdown(f"**Similarity error:** `{error}`"))
        finally:
            score_button.disabled = False

discover_button.on_click(run_discovery)
score_button.on_click(score_nearest)
